In [2]:
!pip install -q evaluate rouge_score sacrebleu nltk

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platfor

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")

In [4]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [5]:
import os
os.environ["WANDB_API_KEY"] = WANDB_API_KEY

import wandb

In [6]:
!rm -rf /kaggle/working/event-planned-story-gen
!git clone https://github.com/abirmondal/event-planned-story-gen.git

Cloning into 'event-planned-story-gen'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 239 (delta 50), reused 92 (delta 36), pack-reused 127 (from 1)
Receiving objects: 100% (239/239), 57.13 MiB | 22.31 MiB/s, done.
Resolving deltas: 100% (105/105), done.
Updating files: 100% (43/43), done.


In [7]:
import sys
from pathlib import Path

# Add the parent directory's path to sys.path
# sys.path requires strings, so we convert the Path object
sys.path.append(str("/kaggle/working/event-planned-story-gen"))

In [32]:
import torch
import numpy as np
import evaluate
from src.dataset_prep.data_for_train import DataForTrain
from src.graph_construction.event_build import EventGraphBuilder
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

In [46]:
data_prep = DataForTrain(
    event_filename_suffix='_event.source_new_2',
    data_types=['test']
)

dataset_dict = data_prep.get_data_for_lc_to_event()

Processing test data:   0%|          | 0/4909 [00:00<?, ?it/s]

In [47]:
for split_name, dataset in dataset_dict.items():
    dataset_dict[split_name] = dataset.shuffle(seed=42)

In [21]:
# Run this code to test the training piepline

# for split_name, dataset in dataset_dict.items():
#     dataset_dict[split_name] =  dataset.select(range(100))

In [42]:
event_graph_read_obj = EventGraphBuilder()

event_graph = event_graph_read_obj.load_graph_pickle('event_graph_new_2.gpickle')
events_list = list(event_graph.nodes)

Graph loaded from /kaggle/working/event-planned-story-gen/data/graph/event_graph_new_2.gpickle.


In [11]:
HF_MODEL_NAME = "abirmondalind/lc-to-event-BART"
# WANDB_RUN_ID = "8o7ji0iz"
# WANDB_PROJECT = "lc-to-event-BART"

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
model = AutoModelForSeq2SeqLM.from_pretrained(HF_MODEL_NAME).to(device)
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
model.eval()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50268, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50268, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_lay

In [14]:
# wandb.init(
#     project=WANDB_PROJECT,
#     id=WANDB_RUN_ID,
#     resume="must"
# )

In [15]:
print("Special Tokens Map:")
tokenizer.special_tokens_map

Special Tokens Map:


{'bos_token': '<s>',
 'eos_token': '</s>',
 'unk_token': '<unk>',
 'sep_token': '</s>',
 'pad_token': '<pad>',
 'cls_token': '<s>',
 'mask_token': '<mask>',
 'additional_special_tokens': ['[EVENT_s]', '[EVENT_sep]', '[EVENT_e]']}

In [16]:
print("\nAdded Tokens Decoder:")
tokenizer.added_tokens_decoder


Added Tokens Decoder:


{0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
 50265: AddedToken("[EVENT_s]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 50266: AddedToken("[EVENT_sep]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 50267: AddedToken("[EVENT_e]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True)}

In [50]:
max_source_length = 256
max_target_length = 64

def preprocess(example):
    inputs = tokenizer(
        example["source"],
        max_length=max_source_length,
        truncation=True,
        padding="max_length"
    )
    targets = tokenizer(
        example["event"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_datasets = dataset_dict.map(
    preprocess,
    batched=True,
    remove_columns=dataset_dict["test"].column_names,
)

Map:   0%|          | 0/19636 [00:00<?, ? examples/s]

In [23]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")


def jaccard_similarity_for_text(text1, text2) -> float:
    """
    Calculate the Jaccard similarity between two texts.

    Args:
        text1 (str): The first text.
        text2 (str): The second text.

    Returns:
        float: The Jaccard similarity between the two texts.
    """
    words1 = set(text1.split())
    words2 = set(text2.split())
    intersection = words1.intersection(words2)
    union = words1.union(words2)
    return len(intersection) / len(union) if len(union) > 0 else 0.0


def get_best_event_from_graph_nodes(test_event: str, events_list: list, return_score: bool = False) -> dict:
    """
    Find the event from the graph with the highest Jaccard similarity to the test event.

    Args:
        test_event (str): The event to compare against.
        events_list (list): A list of events to evaluate.
        return_score (bool): Whether to return the similarity score along with the event.

    Returns:
        
    """
    similarities = []
    for event in events_list:
        similarity = jaccard_similarity_for_text(test_event, event)
        similarities.append((event, similarity))

    max_similarity_event = max(similarities, key=lambda x: x[1])
    if return_score:
        return {"event": max_similarity_event[0], "score": max_similarity_event[1]}
    return {"event": max_similarity_event[0]}


def calculate_metrics_for_events(tokenizer: AutoTokenizer, event_list: list, metrics_prefix: str, use_graph_events: bool = False) -> callable:
    """
    Create a function to compute evaluation metrics for a batch of predictions and labels.

    Args:
        tokenizer: The tokenizer used to decode token IDs to text.
        event_list (list): A list of events from the graph for comparison.
        metrics_prefix (str): A prefix to add to the metric names.
        use_graph_events (bool): Whether to map predicted events to the closest event in the graph.

    Returns:
        function: A function that computes evaluation metrics for a batch of predictions and labels.
    """
    def computer_metrics(pred_events):
        """
        Compute evaluation metrics for a batch of predictions and labels.

        Args:
            pred_events: A tuple containing predicted events and true labels.

        Returns:
            dict: A dictionary containing evaluation metrics.
            - rouge1 (float): ROUGE-1 score.
            - rouge2 (float): ROUGE-2 score.
            - rougeL (float): ROUGE-L score.
            - rougeLsum (float): ROUGE-Lsum score.
            - bleu (float): BLEU score.
            - gen_len (float): Average length of generated events.
        """
        prediction, labels = pred_events

        # Decode the precicted events
        decoded_preds = tokenizer.batch_decode(
            prediction, skip_special_tokens=True)

        # Replace -100 in the labels as we can't decode them
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        # Decode the true events
        decoded_labels = tokenizer.batch_decode(
            labels, skip_special_tokens=True)

        if use_graph_events:
            # Map the decoded predictions to the closest event in the graph
            decoded_preds = [get_best_event_from_graph_nodes(
                pred, event_list)["event"] for pred in decoded_preds]

        # Calculate ROUGE scores
        result = rouge.compute(predictions=decoded_preds,
                               references=decoded_labels, use_stemmer=True)
        result = {f"{metrics_prefix}/{key}": value for key,
                  value in result.items()}

        # Calculate BLEU score
        result[f"{metrics_prefix}/bleu"] = bleu.compute(
            predictions=decoded_preds, references=[[label] for label in decoded_labels])["bleu"]

        # Calculate the average length of the generated events
        result[f"{metrics_prefix}/gen_len"] = np.mean(
            np.count_nonzero(prediction != tokenizer.pad_token_id, axis=1))

        return result

    return computer_metrics

In [24]:
all_preds = []
all_labels = []

In [27]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    
    per_device_eval_batch_size=128,
    fp16=True,

    dataloader_num_workers=2,
    predict_with_generate=True,

    report_to="none",
)

In [28]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
)

In [51]:
raw_predictions = trainer.predict(tokenized_datasets["test"])

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
# predictions = raw_predictions.predictions
# labels = raw_predictions.label_ids

In [23]:
# def calculate_and_log_metrics(predictions, labels):
#     # Decode predictions
#     decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

#     # Decode labels, replacing -100 (which Trainer uses for padding in labels)
#     labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

#     # Compute metrics
#     rouge_scores = rouge.compute(predictions=decoded_preds, references=decoded_labels)
#     list_of_decoded_labels = [[label] for label in decoded_labels]
#     bleu_score = bleu.compute(predictions=decoded_preds, references=list_of_decoded_labels)
#     gen_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]

#     # Prepare metrics for logging
#     metrics = {
#         "test/rouge1": rouge_scores["rouge1"],
#         "test/rouge2": rouge_scores["rouge2"],
#         "test/rougeL": rouge_scores["rougeL"],
#         "test/bleu": bleu_score["bleu"],
#         "test/gen_len": np.mean(gen_lens),
#     }
#     return metrics

In [54]:
import pickle
file_name = "my_bart_prediction_whole.pkl"

with open(file_name, 'wb') as f:
    # 'wb' stands for 'write binary', which is required for pickling
    pickle.dump(raw_predictions, f)

In [52]:
cal_metrices_events_with_graph_comp_fn = calculate_metrics_for_events(tokenizer, events_list, "test_using_graph", use_graph_events=True)
cal_metrices_events_comp_fn = calculate_metrics_for_events(tokenizer, events_list, "test_using_graph", use_graph_events=False)

metrics_with_graph = cal_metrices_events_with_graph_comp_fn((raw_predictions.predictions, raw_predictions.label_ids))
metrics = cal_metrices_events_comp_fn((raw_predictions.predictions, raw_predictions.label_ids))
metrics_with_graph, metrics
# wandb.log(metrics)

KeyboardInterrupt: 

For all samples:
```
test_using_graph/rouge1: 0.18794345691881542
test_using_graph/rouge2: 0.036984951514387125
test_using_graph/rougeL: 0.1872450808906879
test_using_graph/rougeLsum: 0.18729286984151255
test_using_graph/bleu: 0.024515768966569877
test_using_graph/gen_len: 8.08667753106539
```
For 100 samples:
With Graph:
```
'test_using_graph/rouge1': 0.1666349206349207,
'test_using_graph/rouge2': 0.018333333333333333,
'test_using_graph/rougeL': 0.16652777777777783,
'test_using_graph/rougeLsum': 0.16555158730158734,
'test_using_graph/bleu': 0.0,
test_using_graph/gen_len': 8.16
```
Without Graph:
```
'test_using_graph/rouge1': 0.18246428571428572,
'test_using_graph/rouge2': 0.018333333333333333,
'test_using_graph/rougeL': 0.17877380952380953,
'test_using_graph/rougeLsum': 0.17845634920634923,
'test_using_graph/bleu': 0.0,
'test_using_graph/gen_len': 8.16})
```

{'test_using_graph/rouge1': 0.047785714285714286,
 'test_using_graph/rouge2': 0.010666666666666666,
 'test_using_graph/rougeL': 0.04764285714285714,
 'test_using_graph/rougeLsum': 0.04742857142857143,
 'test_using_graph/bleu': 0.0,
 'test_using_graph/gen_len': 8.16}

In [44]:
# wandb.finish()